# Получение метрик с помощью ml flow

## Установка зависимостей

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [6]:
import os
import re
import json
import base64
import subprocess
import time
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

from langchain.agents import create_agent
from langchain.tools import tool

from IPython.display import Image, display, HTML

import mlflow
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer

## Модель и утилиты

In [7]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "atezhelnikova"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "8Zq2EP6H06Yw"
mlflow.set_tracking_uri("https://mlflow.aicorex.tech")
mlflow.set_registry_uri("https://mlflow.aicorex.tech")
mlflow.set_workspace("multi-agent-web-development")
mlflow.set_experiment("demo_08")
mlflow.langchain.autolog()

In [36]:
# ── Настройка директорий ──────────────────────────────────────────────────────
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "../scripts/demo_08"))
DOCKER_DIR = OUTPUT_DIR / "docker"

if OUTPUT_DIR.exists():
    for f in OUTPUT_DIR.iterdir():
        if f.is_file():
            f.unlink()
    print(f"🗑️  Папка очищена: {OUTPUT_DIR}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if DOCKER_DIR.exists():
    for f in DOCKER_DIR.iterdir():
        if f.is_file():
            f.unlink()
    print(f"🗑️  Папка очищена: {DOCKER_DIR}")
DOCKER_DIR.mkdir(parents=True, exist_ok=True)

# ── Модель ────────────────────────────────────────────────────────────────────
model = ChatOpenAI(
    base_url=os.getenv("OPENAI_API_HOST"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model="Qwen/Qwen3.5-27B",
    timeout=120,
    temperature=0.7,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

# ── Утилиты ───────────────────────────────────────────────────────────────────
def save_file(filename: str, content: str) -> str:
    content = re.sub(r'^```[\w]*\n?', '', content.strip())
    content = re.sub(r'\n?```$', '', content.strip())
    filepath = OUTPUT_DIR / filename
    filepath.write_text(content, encoding="utf-8")
    print(f"   💾 Сохранено: {filepath}")
    return str(filepath)

def save_file_docker(filename: str, content: str) -> str:
    content = re.sub(r'^```[\w]*\n?', '', content.strip())
    content = re.sub(r'\n?```$', '', content.strip())
    filepath = DOCKER_DIR / filename
    filepath.write_text(content, encoding="utf-8")
    print(f"   💾 Сохранено: {filepath}")
    return str(filepath)

def run_cmd(cmd: str, cwd: Path = None) -> tuple[int, str, str]:
    result = subprocess.run(
        cmd, shell=True, cwd=cwd,
        capture_output=True, text=True
    )
    return result.returncode, result.stdout.strip(), result.stderr.strip()

# ── Retry-обёртка для вызовов модели ─────────────────────────────────────────
def invoke_with_retry(messages, retries: int = 3, delay: int = 5) -> str:
    """Вызвать модель с повторными попытками при обрыве соединения."""
    for attempt in range(1, retries + 1):
        try:
            result = model.invoke(messages)
            return result.content
        except Exception as e:
            err_str = str(e)
            if attempt < retries and any(x in err_str for x in [
                "Connection error", "RemoteProtocolError",
                "Server disconnected", "APIConnectionError"
            ]):
                print(f"   ⚠️  Обрыв соединения (попытка {attempt}/{retries}), жду {delay}с...")
                time.sleep(delay)
            else:
                raise
    raise RuntimeError("Превышено число попыток подключения к API")

🗑️  Папка очищена: ../scripts/demo_08
🗑️  Папка очищена: ../scripts/demo_08/docker


## Инструменты агентов

In [37]:
# ── Инструменты planner_agent ─────────────────────────────────────────────────

@tool
def create_plan(query: str) -> str:
    """Создать план разработки веб-приложения. Возвращает JSON-список файлов."""
    messages = [
        SystemMessage(content="""Ты — старший планировщик проектов.
Разбей задачу на конкретные файлы. Отвечай ТОЛЬКО валидным JSON массивом.
Каждый элемент: {"filename": "имя файла", "description": "что должно быть внутри"}

ВАЖНО:
- каждый файл ровно один раз
- максимум один js-файл (main.js)
- сначала index.html со всеми id-элементами
- в описаниях css/js указывай какие id/классы использовать из index.html
- в main.js разработчик получит API ключ через инструмент get_api_key
"""),
        HumanMessage(content=query)
    ]
    plan = invoke_with_retry(messages)
    #print(f'{plan=}')
    return plan

In [38]:
# ── Инструменты developer_agent ───────────────────────────────────────────────

@tool
def write_file(description: str) -> str:
    """Написать содержимое одного файла по его описанию. Возвращает готовый код."""
    messages = [
        SystemMessage(content="""Ты — старший веб-разработчик.
Получаешь описание файла — возвращаешь ТОЛЬКО готовое содержимое файла.
Никаких пояснений, никакого markdown."""),
        HumanMessage(content=description)
    ]
    return invoke_with_retry(messages)


@tool
def get_api_key() -> str:
    """
    Получить API ключ для OpenWeatherMap в зашифрованном виде.
    Используй когда нужно вставить API_KEY в main.js.
    """
    raw_key = os.getenv("WEATHER_API_KEY", "")
    if not raw_key:
        return "Ошибка: WEATHER_API_KEY не найден в .env"
    encoded = base64.b64encode(raw_key.encode()).decode()
    js_snippet = f"""// Вставь этот код в начало main.js:
const _k = atob("{encoded}");
// Используй _k вместо API_KEY в запросах к OpenWeatherMap
"""
    print(f"   🔑 [get_api_key] Ключ зашифрован и передан разработчику")
    return js_snippet


@tool
def save_project_file(filename: str, content: str) -> str:
    """Сохранить файл проекта на диск."""
    saved = save_file(filename, content)
    return f"Файл {filename} сохранён: {saved}"

In [39]:
# ── Инструменты executor_agent ────────────────────────────────────────────────

@tool
def write_docker_file(filename: str, description: str) -> str:
    """
    Написать содержимое одного Docker-файла.
    Аргументы: filename — имя файла, description — подробное ТЗ.
    """
    # Dockerfile и docker-compose пишем жёстко — модели не доверяем
    HARDCODED = {
        "Dockerfile": (
            "FROM nginx:1.27-alpine\n"
            "COPY . /usr/share/nginx/html\n"
            "EXPOSE 80\n"
        ),
        "docker-compose.yml": (
            'version: "3.9"\n\n'
            "services:\n"
            "  weather-app:\n"
            "    build:\n"
            "      context: ..\n"
            "      dockerfile: docker/Dockerfile\n"
            "    ports:\n"
            '      - "8080:80"\n'
            "    restart: always\n"
        ),
    }

    if filename in HARDCODED:
        content = HARDCODED[filename]
        print(f"   ✍️  {filename}: использован жёсткий шаблон ({len(content)} символов)")
        return content

    messages = [
        SystemMessage(content="""Ты — Senior DevOps инженер.
Возвращаешь ТОЛЬКО содержимое файла. Никакого markdown, никаких ```."""),
        HumanMessage(content=f"Файл: {filename}\nОписание: {description}")
    ]
    content = invoke_with_retry(messages)
    content = re.sub(r'^```[\w]*\n?', '', content.strip())
    content = re.sub(r'\n?```$', '', content.strip())
    print(f"   ✍️  {filename}: написан ({len(content)} символов)")
    return content


@tool
def save_docker_file(filename: str, content: str) -> str:
    """Сохранить Docker-файл на диск в папку docker."""
    saved = save_file_docker(filename, content)
    return f"Файл {filename} сохранён: {saved}"

In [40]:
# ── Инструменты deploy_agent ──────────────────────────────────────────────────

@tool
def stop_and_remove_containers(placeholder: str = "") -> str:
    """Остановить и удалить все контейнеры текущего проекта. Передай пустую строку."""
    print(f"\n🛑 [Deploy] Останавливаю контейнеры...")
    abs_docker_dir = DOCKER_DIR.resolve()

    code, out, err = run_cmd("docker-compose down --remove-orphans", cwd=abs_docker_dir)
    if code == 0:
        print(f"   ✅ Контейнеры остановлены")
    else:
        print(f"   ⚠️  {err}")

    code2, out2, _ = run_cmd("docker ps -q --filter name=weather-app")
    if out2:
        run_cmd(f"docker rm -f {out2}")
        print(f"   ✅ Принудительно удалён: {out2}")

    return "Контейнеры остановлены и удалены"


@tool
def build_and_run_docker(placeholder: str = "") -> str:
    """Собрать образ и запустить контейнер. Передай пустую строку."""
    print(f"\n🐳 [Deploy] Собираю и запускаю контейнер...")

    abs_docker_dir = DOCKER_DIR.resolve()
    abs_output_dir = OUTPUT_DIR.resolve()
    compose_file = abs_docker_dir / "docker-compose.yml"
    dockerfile = abs_docker_dir / "Dockerfile"

    if not compose_file.exists():
        return f"❌ Ошибка: docker-compose.yml не найден в {abs_docker_dir}"
    if not dockerfile.exists():
        return f"❌ Ошибка: Dockerfile не найден в {abs_docker_dir}"

    print(f"   📂 DOCKER_DIR : {abs_docker_dir}")
    print(f"   📂 OUTPUT_DIR : {abs_output_dir}")
    print(f"   📄 Dockerfile :\n{dockerfile.read_text()}")
    print(f"   📄 docker-compose.yml :\n{compose_file.read_text()}")

    print(f"\n   🔨 Сборка образа...")
    code, out, err = run_cmd("docker-compose build --no-cache", cwd=abs_docker_dir)
    print(f"   stdout: {out[:300]}")
    if code != 0:
        print(f"   stderr: {err[:500]}")
        return f"❌ Ошибка сборки:\n{err}"
    print(f"   ✅ Образ собран")

    print(f"\n   🚀 Запуск контейнера...")
    code, out, err = run_cmd("docker-compose up -d", cwd=abs_docker_dir)
    if code != 0:
        print(f"   stderr: {err[:500]}")
        return f"❌ Ошибка запуска:\n{err}"
    print(f"   ✅ Контейнер запущен")

    code, port_out, _ = run_cmd("docker-compose port weather-app 80", cwd=abs_docker_dir)
    port = port_out.split(":")[-1] if ":" in port_out else "8080"
    url = f"http://localhost:{port}"
    print(f"   🌐 URL: {url}")
    return f"✅ Контейнер запущен. URL: {url}"


@tool
def check_container_status(placeholder: str = "") -> str:
    """Проверить статус контейнеров. Передай пустую строку."""
    code, out, err = run_cmd("docker-compose ps", cwd=DOCKER_DIR.resolve())
    if code != 0:
        return f"Ошибка: {err}"
    return out if out else "Контейнеры не запущены"

## Создание агентов

In [41]:
planner_agent = create_agent(
    model=model,
    tools=[create_plan],
    system_prompt="Ты — агент-планировщик. Вызови create_plan и верни JSON-план файлов проекта.",
    name="planner_agent",
)

developer_agent = create_agent(
    model=model,
    tools=[write_file, save_project_file, get_api_key],
    system_prompt="""Ты — агент-разработчик. Получаешь план в виде JSON-списка файлов.

ОБЯЗАТЕЛЬНЫЙ ПЕРВЫЙ ШАГ ДО ВСЕГО:
- Вызови get_api_key() — получи зашифрованный JS-сниппет с ключом
- Сохрани его, он понадобится для main.js

Для каждого файла из плана:
1. Вызови write_file с описанием файла
   - Если это main.js — ОБЯЗАТЕЛЬНО включи в описание полученный js_snippet из get_api_key
   - Напомни: переменная _k содержит API ключ, использовать именно её в fetch запросах
2. Вызови save_project_file с именем файла и содержимым

Обработай ВСЕ файлы из плана.""",
    name="developer_agent",
)

executor_agent = create_agent(
    model=model,
    tools=[write_docker_file, save_docker_file],
    system_prompt="""Ты — DevOps-разработчик. Создай файлы для Docker-деплоя.

Создай СТРОГО ЭТИ файлы по порядку, для каждого:
1. Вызови write_docker_file(filename, description)
2. Вызови save_docker_file(filename, content)

Список файлов:
- filename: "Dockerfile"
  description: "Контейнер для раздачи статических файлов (html, css, js).
  Контекст сборки — папка с фронтендом."

- filename: "docker-compose.yml"
  description: "Запуск контейнера weather-app, порт 8080,
  context: .. , dockerfile: docker/Dockerfile, restart: always"

- filename: ".env.example"
  description: "OPENAI_API_HOST, OPENAI_API_KEY, WEATHER_API_KEY с пустыми значениями"

- filename: ".dockerignore"
  description: ".env, .git, __pycache__, *.pyc, docker/"

Обработай ВСЕ 4 файла.""",
    name="executor_agent",
)

deploy_agent = create_agent(
    model=model,
    tools=[stop_and_remove_containers, build_and_run_docker, check_container_status],
    system_prompt="""Ты — агент деплоя. Выполни строго по порядку:
1. stop_and_remove_containers("") — останови старые контейнеры
2. build_and_run_docker("") — собери и запусти новый контейнер
3. check_container_status("") — проверь что контейнер работает
Верни итоговый URL приложения.""",
    name="deploy_agent",
)

## Supervisor 

In [42]:
@tool
def run_planner(task: str) -> str:
    """Запустить агента-планировщика. Составляет JSON-план файлов веб-проекта.
    Передай задачу пользователя целиком."""
    result = planner_agent.invoke(
        {"messages": [{"role": "user", "content": task}]},
        config={"recursion_limit": 20}
    )
    output = result["messages"][-1].content
    print(f"\n📋 [Planner] завершён")
    return output


@tool
def run_developer(plan: str) -> str:
    """Запустить агента-разработчика. Пишет и сохраняет все файлы фронтенда.
    Передай JSON-план от планировщика."""
    result = developer_agent.invoke(
        {"messages": [{"role": "user", "content": plan}]},
        config={"recursion_limit": 30}
    )
    output = result["messages"][-1].content
    print(f"\n👨‍💻 [Developer] завершён")
    return output


@tool
def run_executor(task: str = "Создай Docker-файлы для деплоя") -> str:
    """Запустить DevOps-агента. Создаёт Dockerfile, docker-compose.yml и вспомогательные файлы.
    Всегда передавай строку с задачей."""
    result = executor_agent.invoke(
        {"messages": [{"role": "user", "content": task}]},
        config={"recursion_limit": 20}
    )
    output = result["messages"][-1].content
    print(f"\n🔧 [Executor] завершён")
    return output


@tool
def run_deploy(task: str = "Задеплой приложение") -> str:
    """Запустить агента деплоя. Останавливает старые контейнеры, собирает образ и запускает новый.
    Возвращает URL приложения."""
    result = deploy_agent.invoke(
        {"messages": [{"role": "user", "content": task}]},
        config={"recursion_limit": 20}
    )
    output = result["messages"][-1].content
    print(f"\n🚀 [Deploy] завершён")
    return output

In [43]:
supervisor = create_agent(
    model=model,
    tools=[run_planner, run_developer, run_executor, run_deploy],
    system_prompt="""Ты — менеджер команды разработки. Выполняй СТРОГО по порядку:

1. run_planner      — составит план файлов проекта
2. run_developer    — напишет и сохранит файлы фронтенда (передай ему план от планировщика)
3. run_executor     — создаст Dockerfile и docker-compose.yml
4. run_deploy       — остановит старые контейнеры, соберёт и запустит новый

НЕ пропускай ни одного агента.
НЕ завершай работу пока все четыре агента не выполнены.
В финальном ответе обязательно укажи URL приложения.""",
    name="supervisor",
)

## Запуск

In [45]:
query = """Создай веб-приложение, которое показывает погоду в Москве
на ближайшие 3 дня. Данные бери с openweathermap.org."""

print("\nЗапуск мультиагентной системы (LangChain Subagents)...")
print("=" * 60)

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"recursion_limit": 80}
)

# ── Итог ──────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("\n✅ Supervisor завершил работу:")
print(result["messages"][-1].content)

saved = list(OUTPUT_DIR.iterdir())
if saved:
    print("\n📁 Созданные файлы проекта:")
    for f in sorted(saved):
        if f.is_file():
            print(f"   • {f.name}")

saved_docker = list(DOCKER_DIR.iterdir())
if saved_docker:
    print("\n🐳 Docker-файлы:")
    for f in sorted(saved_docker):
        print(f"   • {f.name}")

# ── Финальный статус и ссылка ─────────────────────────────────────────────────
print("\n" + "=" * 60)
_, status, _ = run_cmd("docker-compose ps", cwd=DOCKER_DIR.resolve())
print(f"\n📊 Статус контейнеров:\n{status}")

_, port_out, _ = run_cmd("docker-compose port weather-app 80", cwd=DOCKER_DIR.resolve())
port = port_out.split(":")[-1] if ":" in port_out else "8080"
url = f"http://localhost:{port}"
print(f"\n🌐 Приложение доступно: {url}")
display(HTML(f'<h2>🌤️ <a href="{url}" target="_blank">Открыть приложение: {url}</a></h2>'))


Запуск мультиагентной системы (LangChain Subagents)...

📋 [Planner] завершён
   🔑 [get_api_key] Ключ зашифрован и передан разработчику
   💾 Сохранено: ../scripts/demo_08/index.html
   💾 Сохранено: ../scripts/demo_08/styles.css
   💾 Сохранено: ../scripts/demo_08/main.js

👨‍💻 [Developer] завершён
   ✍️  Dockerfile: использован жёсткий шаблон (62 символов)
   💾 Сохранено: ../scripts/demo_08/docker/Dockerfile
   ✍️  docker-compose.yml: использован жёсткий шаблон (155 символов)
   💾 Сохранено: ../scripts/demo_08/docker/docker-compose.yml
   ✍️  .env.example: написан (49 символов)
   💾 Сохранено: ../scripts/demo_08/docker/.env.example
   ✍️  .dockerignore: написан (35 символов)
   💾 Сохранено: ../scripts/demo_08/docker/.dockerignore

🔧 [Executor] завершён

🛑 [Deploy] Останавливаю контейнеры...
   ✅ Контейнеры остановлены

🐳 [Deploy] Собираю и запускаю контейнер...
   📂 DOCKER_DIR : /Users/annatezelnikova/Desktop/Agents/scripts/demo_08/docker
   📂 OUTPUT_DIR : /Users/annatezelnikova/Desktop/

Trace(trace_id=tr-3ecfa479dbcfaa2975fd6b41cb577421)

In [46]:
@scorer
def expert_code_review(outputs) -> Feedback:
    """Судья просит модель сделать expert code review"""
    code = str(outputs)
    
    if not code or len(code) < 100:
        return Feedback(value=0.0, rationale="Code too short")
    
    try:
        prompt = f"""Ты опытный разработчик. Проанализируй этот веб-код:

{code[:2000]}

Оцени по шкале 0-10:
1. Читаемость кода
2. Безопасность
3. Производительность
4. Следование best practices

JSON: {{"readability": 8, "security": 7, "performance": 8, "best_practices": 7}}

ТОЛЬКО JSON!"""
        
        response = judge_model.invoke(prompt)
        response_text = response.content.strip()
        
        try:
            scores = json.loads(response_text)
            avg_score = (
                scores.get("readability", 0) +
                scores.get("security", 0) +
                scores.get("performance", 0) +
                scores.get("best_practices", 0)
            ) / 4 / 10
            
            SCORES_CACHE['expert_code_review'] = avg_score
            return Feedback(
                value=avg_score,
                rationale=f"Expert review: Read:{scores.get('readability')}/10, "
                         f"Sec:{scores.get('security')}/10, "
                         f"Perf:{scores.get('performance')}/10, "
                         f"BP:{scores.get('best_practices')}/10"
            )
        except json.JSONDecodeError:
            return Feedback(value=0.6, rationale=f"Model: {response_text[:100]}")
    
    except Exception as e:
        return Feedback(value=0.5, rationale=f"Error: {str(e)[:50]}")

In [47]:
# @scorer
# def llm_functionality_check(inputs, outputs) -> Feedback:
#     """Проверка функциональности"""
#     if not inputs or not outputs:
#         return Feedback(value=0.5, rationale="No data")
    
#     requirement = str(inputs)
#     code = str(outputs)
    
#     try:
#         prompt = f"""Проверь соответствие кода требованиям.

# ТРЕБОВАНИЕ: {requirement}

# КОД: {code[:1500]}

# Вопросы:
# 1. Реализует ли код основные требования? (да/нет)
# 2. Есть ли потенциальные баги? (да/нет)
# 3. Код будет работать как ожидается? (да/нет)

# JSON: {{"meets_requirements": true, "has_bugs": false, "will_work": true, "confidence": 0.9}}

# ТОЛЬКО JSON!"""
        
#         response = judge_model.invoke(prompt)
#         response_text = response.content.strip()
        
#         try:
#             result = json.loads(response_text)
#             score = 0.0
#             if result.get("meets_requirements"):
#                 score += 0.4
#             if not result.get("has_bugs"):
#                 score += 0.3
#             if result.get("will_work"):
#                 score += 0.3
            
#             confidence = result.get("confidence", 0.0)
#             final_score = score * confidence
            
#             SCORES_CACHE['llm_functionality_check'] = final_score
#             return Feedback(
#                 value=final_score,
#                 rationale=f"Функция: {'✓' if result.get('meets_requirements') else '✗'}, "
#                          f"Баги: {'✗' if result.get('has_bugs') else '✓'}, "
#                          f"Работает: {'✓' if result.get('will_work') else '✗'}"
#             )
#         except:
#             return Feedback(value=0.5, rationale=response_text[:80])
    
#     except Exception as e:
#         return Feedback(value=0.5, rationale=f"Error: {str(e)[:50]}")

In [48]:
# @scorer
# def llm_functionality_check(inputs, outputs) -> Feedback:
#     """Проверка функциональности"""
#     # ── ДИАГНОСТИКА ──────────────────────────────────────────────────────────
#     print('\n' + '='*60)
#     print('🔍 [llm_functionality_check] СТАРТ')
#     print(f'   inputs type  : {type(inputs)}')
#     print(f'   inputs value : {str(inputs)[:200]}')
#     print(f'   outputs type : {type(outputs)}')
#     print(f'   outputs value: {str(outputs)[:200]}')
#     print('='*60)

#     if not inputs or not outputs:
#         print('   ❌ ПРИЧИНА НУЛЯ: inputs или outputs пустые!')
#         return Feedback(value=0.5, rationale='No data')

#     requirement = str(inputs)
#     code = str(outputs)

#     print(f'   requirement (200 chars): {requirement[:200]}')
#     print(f'   code length: {len(code)}')

#     try:
#         prompt = f"""Проверь соответствие кода требованиям.

# ТРЕБОВАНИЕ: {requirement}

# КОД: {code[:1500]}

# Вопросы:
# 1. Реализует ли код основные требования? (да/нет)
# 2. Есть ли потенциальные баги? (да/нет)
# 3. Код будет работать как ожидается? (да/нет)

# JSON: {{"meets_requirements": true, "has_bugs": false, "will_work": true, "confidence": 0.9}}

# ТОЛЬКО JSON!"""

#         print('   📤 Отправляю запрос к judge_model...')
#         response = judge_model.invoke(prompt)
#         response_text = response.content.strip()
#         print(f'   📥 Ответ модели: {response_text[:300]}')

#         try:
#             result = json.loads(response_text)
#             print(f'   ✅ JSON распарсен: {result}')

#             score = 0.0
#             if result.get('meets_requirements'):
#                 score += 0.4
#                 print('   +0.4 meets_requirements = True')
#             else:
#                 print('   +0.0 meets_requirements = False/None')

#             if not result.get('has_bugs'):
#                 score += 0.3
#                 print('   +0.3 has_bugs = False (нет багов)')
#             else:
#                 print('   +0.0 has_bugs = True (есть баги)')

#             if result.get('will_work'):
#                 score += 0.3
#                 print('   +0.3 will_work = True')
#             else:
#                 print('   +0.0 will_work = False/None')

#             confidence = result.get('confidence', 0.0)
#             final_score = score * confidence
#             print(f'   score={score:.2f}, confidence={confidence}, final={final_score:.3f}')

#             if final_score == 0.0:
#                 print('   ⚠️  ПРИЧИНА НУЛЯ: score или confidence равны 0!')
#                 print(f'      meets_req={result.get("meets_requirements")}, '
#                       f'has_bugs={result.get("has_bugs")}, '
#                       f'will_work={result.get("will_work")}')
#                 print(f'      confidence raw value: {result.get("confidence")}')

#             SCORES_CACHE['llm_functionality_check'] = final_score
#             return Feedback(
#                 value=final_score,
#                 rationale=f"Функция: {'✓' if result.get('meets_requirements') else '✗'}, "
#                          f"Баги: {'✗' if result.get('has_bugs') else '✓'}, "
#                          f"Работает: {'✓' if result.get('will_work') else '✗'} | "
#                          f"confidence={confidence:.2f} | raw_score={score:.2f}"
#             )
#         except Exception as parse_err:
#             print(f'   ❌ ПРИЧИНА НУЛЯ: JSON parse error: {parse_err}')
#             print(f'   Сырой ответ модели: {response_text[:300]}')
#             return Feedback(value=0.5, rationale=f'JSON parse failed: {response_text[:80]}')

#     except Exception as e:
#         print(f'   ❌ ПРИЧИНА НУЛЯ: Exception в invoke: {str(e)[:200]}')
#         import traceback
#         traceback.print_exc()
#         return Feedback(value=0.5, rationale=f'Error: {str(e)[:50]}')

In [62]:
# ── Ручной тест llm_functionality_check ──────────────────────────────────────
print("Запускаю llm_functionality_check вручную...")

# Берём реальные данные из eval_data который уже собран выше
test_inputs  = {"query": query}
test_outputs = {
    "response": "\n".join(generated_files.values()),
    "files": generated_files,
}

result = llm_functionality_check(
    inputs=test_inputs,
    outputs=test_outputs,
)

print(f"\n📊 Результат:")
print(f"   value    : {result.value}")
# print(f"   rationale: {result.rationale}")

Запускаю llm_functionality_check вручную...

📊 Результат:
   value    : 0.0


Trace(trace_id=tr-be3e016090b4ead5c98e15a854793b96)

In [63]:
# Посмотрим что именно судья считает багом
prompt_debug = f"""Проверь соответствие кода требованиям.

ТРЕБОВАНИЕ: {query}

КОД: {list(generated_files.values())[0][:3000]}

Что именно не так? Перечисли конкретные проблемы.
Ответь свободным текстом."""

response = judge_model.invoke(prompt_debug)
# print(response.content)

Trace(trace_id=tr-3ac95a8eda10dc883e10b5ce31e37f99)

In [51]:
@scorer
def llm_functionality_check(inputs, outputs) -> Feedback:
    """Соответствие требованиям, наличие багов, работоспособность."""
    if not inputs or not outputs:
        return Feedback(value=0.5, rationale="No data")

    # inputs приходит как словарь {"query": "..."} — достаём строку
    if isinstance(inputs, dict):
        requirement = inputs.get("query", str(inputs))
    else:
        requirement = str(inputs)

    code = str(outputs)

    try:
        prompt = f"""Проверь соответствие кода требованиям.

ТРЕБОВАНИЕ: {requirement}

КОД: {code[:1500]}

Вопросы:
1. Реализует ли код основные требования? (да/нет)
2. Есть ли потенциальные баги? (да/нет)
3. Код будет работать как ожидается? (да/нет)

JSON: {{"meets_requirements": true, "has_bugs": false, "will_work": true, "confidence": 0.9, "reason": "код корректно реализует требования, API ключ подставлен"}}


ТОЛЬКО JSON!"""

        response = judge_model.invoke(prompt)
        response_text = response.content.strip()

        try:
            result = json.loads(response_text)
            #score = 0.0
            if result.get("meets_requirements"):
                score += 0.4
            if not result.get("has_bugs"):
                score += 0.3
            if result.get("will_work"):
                score += 0.3

            # если модель не вернула confidence — считаем 1.0 (полная уверенность)
            confidence = result.get("confidence", 1.0)
            final_score = score * confidence

            SCORES_CACHE['llm_functionality_check'] = final_score
            return Feedback(
                        value=final_score,
                        rationale=f"Функция: {'✓' if result.get('meets_requirements') else '✗'}, "
                                 f"Баги: {'✗' if result.get('has_bugs') else '✓'}, "
                                 f"Работает: {'✓' if result.get('will_work') else '✗'} | "
                                 f"confidence={confidence:.2f} | "
                                 f"Причина: {result.get('reason', 'не указана')}"
                    )
        except json.JSONDecodeError:
            return Feedback(value=0.5, rationale=f"JSON parse failed: {response_text[:80]}")

    except Exception as e:
        return Feedback(value=0.5, rationale=f"Error: {str(e)[:50]}")

In [52]:
@scorer
def llm_architecture_assessment(outputs) -> Feedback:
    """
    Оценка архитектуры и структуры кода
    """
    
    code = str(outputs)
    
    if not code or len(code) < 200:
        return Feedback(value=0.3, rationale="Code too short for architecture assessment")
    
    try:
        prompt = f"""Ты архитектор. Оцени архитектуру этого кода:

{code[:2000]}

Оцени (1-10):
- Модульность (разделение на части)
- Масштабируемость (легко ли добавлять новое)
- Поддерживаемость (легко ли менять/исправлять)
- Повторное использование (DRY принцип)

JSON формат:
{{"modularity": 7, "scalability": 6, "maintainability": 8, "reusability": 7}}

ТОЛЬКО JSON!"""
        
        response = model.invoke(prompt)
        response_text = response.content.strip()
        
        try:
            scores = json.loads(response_text)
            
            avg = (
                scores.get("modularity", 0) +
                scores.get("scalability", 0) +
                scores.get("maintainability", 0) +
                scores.get("reusability", 0)
            ) / 4 / 10
            
            SCORES_CACHE['llm_architecture_assessment'] = avg
            return Feedback(
                value=avg,
                rationale=f"Architecture: Modularity {scores.get('modularity')}/10, "
                         f"Scalability {scores.get('scalability')}/10, "
                         f"Maintainability {scores.get('maintainability')}/10, "
                         f"Reusability {scores.get('reusability')}/10"
            )
        except json.JSONDecodeError:
            return Feedback(value=0.5, rationale=response_text[:100])
    
    except Exception as e:
        return Feedback(value=0.5, rationale=f"Error: {str(e)[:50]}")


In [53]:
@scorer
def docker_build_assessment(outputs) -> Feedback:
    """
    Оценка Docker-сборки: качество Dockerfile/docker-compose + время сборки.
    Ожидает в outputs ключи 'docker_files' и 'build_time_seconds' (опционально).
    """
    import time as _time

    docker_files = {}
    build_time = None

    if isinstance(outputs, dict):
        docker_files = outputs.get("docker_files", {})
        build_time = outputs.get("build_time_seconds")

    # Если docker_files не переданы — читаем с диска
    if not docker_files and DOCKER_DIR.exists():
        for fp in DOCKER_DIR.iterdir():
            if fp.is_file():
                try:
                    docker_files[fp.name] = fp.read_text(encoding="utf-8")
                except Exception:
                    pass

    if not docker_files:
        return Feedback(value=0.0, rationale="Docker files not found")

    # ----- LLM-оценка качества Docker-файлов -----
    files_text = "\n\n".join(
        f"=== {name} ===\n{content}"
        for name, content in docker_files.items()
    )

    try:
        prompt = (
            "Ты DevOps-эксперт. Оцени эти Docker-файлы по шкале 0-10:\n\n"
            + files_text[:3000]
            + "\n\nКритерии:\n"
            "- correctness: корректность синтаксиса и конфигурации\n"
            "- security: безопасность (нет root, минимальный образ)\n"
            "- best_practices: best practices (layer caching, .dockerignore, restart)\n"
            "- completeness: наличие всех нужных файлов\n\n"
            'ТОЛЬКО JSON: {"correctness": 8, "security": 6, "best_practices": 7, "completeness": 9}'
        )
        response = judge_model.invoke(prompt)
        scores = json.loads(response.content.strip())
        quality = (
            scores.get("correctness", 0)
            + scores.get("security", 0)
            + scores.get("best_practices", 0)
            + scores.get("completeness", 0)
        ) / 4 / 10
    except Exception as e:
        quality = 0.5
        scores = {}

    # ----- Оценка времени сборки -----
    if build_time is None:
        t0 = _time.time()
        code, _, err = run_cmd("docker-compose build --no-cache", cwd=DOCKER_DIR.resolve())
        build_time = _time.time() - t0
        build_ok = (code == 0)
    else:
        build_ok = True

    # Нормализуем: <30 сек → 1.0, >300 сек → 0.0
    time_score = max(0.0, min(1.0, 1.0 - (build_time - 30) / 270))
    build_status = "✅ успешно" if build_ok else "❌ ошибка"

    final = quality * 0.7 + time_score * 0.3

    rationale = (
        f"Docker quality: Corr {scores.get('correctness','?')}/10, "
        f"Sec {scores.get('security','?')}/10, "
        f"BP {scores.get('best_practices','?')}/10, "
        f"Compl {scores.get('completeness','?')}/10 | "
        f"Build: {build_status} за {build_time:.1f}с"
    )
    SCORES_CACHE['docker_build_assessment'] = round(final, 3)
    return Feedback(value=round(final, 3), rationale=rationale)


In [54]:
@scorer
def webpage_quality_assessment(outputs) -> Feedback:
    """
    Оценка качества веб-страницы созданной моделью (Gemma-судья).
    Оценивает HTML/CSS/JS файлы из OUTPUT_DIR.
    """
    web_files = {}
    if isinstance(outputs, dict):
        files = outputs.get("files", {})
        web_files = {k: v for k, v in files.items()
                     if k.endswith((".html", ".css", ".js"))}

    if not web_files and OUTPUT_DIR.exists():
        for fp in OUTPUT_DIR.iterdir():
            if fp.is_file() and fp.suffix in (".html", ".css", ".js"):
                try:
                    web_files[fp.name] = fp.read_text(encoding="utf-8")
                except Exception:
                    pass

    if not web_files:
        return Feedback(value=0.0, rationale="No web files found")

    # Gemma-судья (пока Гвен, потом поменяем на Гемму)
    gemma_judge = ChatOpenAI(
        base_url=os.getenv("OPENAI_API_HOST"),
        api_key=os.getenv("OPENAI_API_KEY"),
        model="Qwen/Qwen3.5-27B",
        temperature=0.3,
    )

    files_text = "\n\n".join(
        f"=== {name} ===\n{content[:1500]}"
        for name, content in web_files.items()
    )

    try:
        prompt = (
            "Ты UX/frontend эксперт. Оцени веб-страницу по шкале 0-10:\n\n"
            + files_text[:4000]
            + "\n\nКритерии:\n"
            "- ui_design: визуальное оформление и структура\n"
            "- ux_usability: удобство использования\n"
            "- responsiveness: адаптивность (mobile-friendly)\n"
            "- functionality: правильность реализации функционала\n"
            "- code_quality: чистота и читаемость кода\n\n"
            'ТОЛЬКО JSON: {"ui_design": 7, "ux_usability": 8, "responsiveness": 6, "functionality": 8, "code_quality": 7}'
        )
        response = gemma_judge.invoke(prompt)
        scores = json.loads(response.content.strip())

        avg = (
            scores.get("ui_design", 0)
            + scores.get("ux_usability", 0)
            + scores.get("responsiveness", 0)
            + scores.get("functionality", 0)
            + scores.get("code_quality", 0)
        ) / 5 / 10

        rationale = (
            f"[Gemma judge] UI {scores.get('ui_design','?')}/10, "
            f"UX {scores.get('ux_usability','?')}/10, "
            f"Responsive {scores.get('responsiveness','?')}/10, "
            f"Func {scores.get('functionality','?')}/10, "
            f"Code {scores.get('code_quality','?')}/10"
        )
        SCORES_CACHE['webpage_quality_assessment'] = round(avg, 3)
        return Feedback(value=round(avg, 3), rationale=rationale)

    except Exception as e:
        return Feedback(value=0.5, rationale=f"Gemma judge error: {str(e)[:80]}")


In [55]:
# ── Словарь для хранения времени работы агентов ─────────────────────────────
AGENT_TIMINGS: dict = {}  # {agent_name: elapsed_seconds}


def timed_run(agent_name: str, agent_fn, *args, **kwargs):
    """Обёртка: запускает агента и сохраняет время в AGENT_TIMINGS."""
    import time as _time
    t0 = _time.time()
    result = agent_fn(*args, **kwargs)
    elapsed = _time.time() - t0
    AGENT_TIMINGS[agent_name] = round(elapsed, 2)
    print(f"   ⏱️  {agent_name}: {elapsed:.1f}с")
    return result


@scorer
def agent_timing_metric(outputs) -> Feedback:
    """
    Метрика времени работы каждого агента.
    Считывает AGENT_TIMINGS, логирует каждое время в MLflow.
    Хорошо: агент < 60 сек → вклад 1.0; плохо: > 300 сек → вклад 0.0.
    """
    if not AGENT_TIMINGS:
        return Feedback(value=0.5, rationale="No agent timing data. Use timed_run() wrapper.")

    scores = []
    parts = []
    for agent_name, elapsed in AGENT_TIMINGS.items():
        s = max(0.0, min(1.0, 1.0 - (elapsed - 60) / 240))
        scores.append(s)
        parts.append(f"{agent_name}: {elapsed:.1f}с")
        try:
            mlflow.log_metric(f"agent_time_{agent_name}", elapsed)
        except Exception:
            pass

    avg_score = sum(scores) / len(scores)
    total_time = sum(AGENT_TIMINGS.values())
    rationale = "Времена агентов: " + ", ".join(parts) + f" | Итого: {total_time:.1f}с"
    SCORES_CACHE['agent_timing_metric'] = round(avg_score, 3)
    return Feedback(value=round(avg_score, 3), rationale=rationale)


print("✓ agent_timing_metric и timed_run зарегистрированы")
print("  Пример: timed_run('planner', run_planner, task)")


✓ agent_timing_metric и timed_run зарегистрированы
  Пример: timed_run('planner', run_planner, task)


In [56]:
@scorer
def test_results_metric(outputs) -> Feedback:
    """
    Метрика по результатам тестов: пройдено / не пройдено.
    Запускает pytest на OUTPUT_DIR или базовые smoke-тесты.
    """
    import subprocess
    import re as _re

    test_files = []
    if OUTPUT_DIR.exists():
        test_files = list(OUTPUT_DIR.glob("test_*.py")) + list(OUTPUT_DIR.glob("*_test.py"))

    passed = 0
    failed = 0
    errors = 0

    if test_files:
        result = subprocess.run(
            ["python", "-m", "pytest", "--tb=no", "-q", str(OUTPUT_DIR)],
            capture_output=True, text=True
        )
        output = result.stdout + result.stderr
        m_passed = _re.search(r"(\d+) passed", output)
        m_failed = _re.search(r"(\d+) failed", output)
        m_error  = _re.search(r"(\d+) error",  output)
        passed = int(m_passed.group(1)) if m_passed else 0
        failed = int(m_failed.group(1)) if m_failed else 0
        errors = int(m_error.group(1))  if m_error  else 0
    else:
        # Smoke-тесты: проверяем ключевые файлы и контейнер
        checks = [
            ("index.html exists",     (OUTPUT_DIR / "index.html").exists()),
            ("Dockerfile exists",     (DOCKER_DIR / "Dockerfile").exists()),
            ("docker-compose exists", (DOCKER_DIR / "docker-compose.yml").exists()),
        ]
        code, out, _ = run_cmd("docker-compose ps", cwd=DOCKER_DIR.resolve())
        checks.append(("container running", "Up" in out))

        for name, ok in checks:
            if ok:
                passed += 1
            else:
                failed += 1

    total = passed + failed + errors
    if total == 0:
        return Feedback(value=0.0, rationale="No tests found or run")

    score = passed / total

    try:
        mlflow.log_metric("tests_passed", passed)
        mlflow.log_metric("tests_failed", failed)
        mlflow.log_metric("tests_errors", errors)
        mlflow.log_metric("tests_total",  total)
    except Exception:
        pass

    rationale = f"Тесты: ✅ {passed} пройдено / ❌ {failed} провалено / ⚠️ {errors} ошибок | Итого: {total}"
    SCORES_CACHE['test_results_metric'] = round(score, 3)
    return Feedback(value=round(score, 3), rationale=rationale)


In [57]:
@scorer
def llm_overall_assessment(inputs, outputs) -> Feedback:
    """
    Агрегированная итоговая оценка проекта.
    Собирает баллы всех остальных метрик из глобального словаря SCORES_CACHE,
    взвешивает их и просит LLM дать финальный комментарий с учётом всей картины.
    """
    # ── Весовые коэффициенты для каждой метрики ──────────────────────────────
    WEIGHTS = {
        "expert_code_review":        0.20,
        "llm_functionality_check":   0.20,
        "llm_architecture_assessment": 0.15,
        "docker_build_assessment":   0.15,
        "webpage_quality_assessment": 0.15,
        "agent_timing_metric":       0.08,
        "test_results_metric":       0.07,
    }

    # ── Попытка собрать баллы из глобального кэша (заполняется другими скорерами)
    scores_used = {}
    weighted_sum = 0.0
    total_weight = 0.0

    cache = globals().get('SCORES_CACHE', {})
    for metric, weight in WEIGHTS.items():
        if metric in cache:
            scores_used[metric] = cache[metric]
            weighted_sum += cache[metric] * weight
            total_weight += weight

    # Если кэш пуст — падаем обратно на LLM-оценку без агрегации
    if not scores_used:
        code = str(outputs)[:2000] if outputs else 'No code'
        requirement = str(inputs) if inputs else 'No requirement'
        try:
            prompt = (
                "Ты senior разработчик. Оцени проект от 0 до 100.\n\n"
                f"ТРЕБОВАНИЕ:\n{requirement}\n\nКОД:\n{code}\n\n"
                'ТОЛЬКО JSON: {"overall_score": 75, "summary": "...", "main_issues": "..."}'
            )
            response = judge_model.invoke(prompt)
            result = json.loads(response.content.strip())
            score = result.get('overall_score', 50) / 100
            return Feedback(
                value=round(score, 3),
                rationale=(
                    f"[Fallback LLM] {result.get('summary', '')} | "
                    f"Issues: {result.get('main_issues', 'None')}"
                )
            )
        except Exception as e:
            return Feedback(value=0.5, rationale=f'Fallback error: {str(e)[:60]}')

    # ── Взвешенный итог ───────────────────────────────────────────────────────
    aggregate = weighted_sum / total_weight if total_weight > 0 else 0.0

    # ── LLM финальный комментарий с учётом всех баллов ───────────────────────
    scores_str = '\n'.join(
        f'  {k}: {v:.0%}' for k, v in scores_used.items()
    )
    code_snippet = str(outputs)[:1000] if outputs else ''
    requirement = str(inputs)[:500] if inputs else 'No requirement'

    try:
        prompt = (
            "Ты senior разработчик. Дай финальный вывод по проекту.\n\n"
            f"ТРЕБОВАНИЕ:\n{requirement}\n\n"
            f"БАЛЛЫ МЕТРИК:\n{scores_str}\n\n"
            f"ФРАГМЕНТ КОДА:\n{code_snippet}\n\n"
            "Учти все баллы выше и дай краткий синтез: что хорошо, что надо исправить в первую очередь.\n"
            'ТОЛЬКО JSON: {"summary": "...", "priority_fix": "...", "verdict": "production_ready|needs_work|critical_issues"}'
        )
        response = judge_model.invoke(prompt)
        result = json.loads(response.content.strip())
        rationale = (
            f"Aggregate {aggregate:.0%} (weighted) | "
            f"Verdict: {result.get('verdict', '?')} | "
            f"{result.get('summary', '')} | "
            f"Fix first: {result.get('priority_fix', 'None')}"
        )
    except Exception as e:
        rationale = f'Aggregate {aggregate:.0%} (weighted) | Comment error: {str(e)[:60]}'

    # Логируем агрегат в MLflow
    try:
        mlflow.log_metric('overall_weighted_score', aggregate)
    except Exception:
        pass

    return Feedback(value=round(aggregate, 3), rationale=rationale)


# ── Глобальный кэш для передачи баллов между скорерами ───────────────────────
# Остальные скореры должны писать в него: SCORES_CACHE['expert_code_review'] = score
SCORES_CACHE: dict = {}
print('✓ llm_overall_assessment переписан как агрегатор')
print('  Заполняй SCORES_CACHE в других скорерах для агрегации')

✓ llm_overall_assessment переписан как агрегатор
  Заполняй SCORES_CACHE в других скорерах для агрегации


In [58]:
LLM_JUDGES = [
    expert_code_review,            # читаемость, безопасность, производительность, best practices кода
    llm_functionality_check,       # соответствие требованиям, наличие багов, работоспособность
    llm_architecture_assessment,   # модульность, масштабируемость, поддерживаемость, DRY
    llm_overall_assessment,        # взвешенный агрегат всех метрик + финальный verdict LLM
    docker_build_assessment,       # качество Docker-файлов (LLM) + время сборки образа
    webpage_quality_assessment,    # UI/UX, адаптивность, функционал страницы (Gemma-судья)
    agent_timing_metric,           # время работы каждого агента, логируется в MLflow
    test_results_metric,           # пройдено/провалено тестов (pytest или smoke-тесты)
]

print("\n✓ Загружено LLM судей:", len(LLM_JUDGES))
for j in LLM_JUDGES:
    name = getattr(j, 'name', None) or getattr(j, '__name__', str(j))
    print(f"  • {name}")



✓ Загружено LLM судей: 8
  • expert_code_review
  • llm_functionality_check
  • llm_architecture_assessment
  • llm_overall_assessment
  • docker_build_assessment
  • webpage_quality_assessment
  • agent_timing_metric
  • test_results_metric


In [59]:
print("\n" + "="*100)
print("ПОДГОТОВКА ДАННЫХ ДЛЯ СУДЕЙ")
print("="*100)

# Собрать файлы из OUTPUT_DIR
generated_files = {}

if OUTPUT_DIR.exists():
    print(f"\n📂 Папка: {OUTPUT_DIR}")
    for file_path in sorted(OUTPUT_DIR.iterdir()):
        if file_path.is_file():
            try:
                content = file_path.read_text(encoding='utf-8')
                generated_files[file_path.name] = content
                print(f"  ✓ {file_path.name:20} | {len(content):6} символов")
            except Exception as e:
                print(f"  ❌ {file_path.name}: {e}")

# Объединить весь код
all_code = "\n\n".join([
    f"{'='*60}\n" + 
    f"FILE: {filename}\n" +
    f"{'='*60}\n" +
    content
    for filename, content in generated_files.items()
])

print(f"\n✓ Всего файлов: {len(generated_files)}")
print(f"✓ Общий размер кода: {len(all_code):,} символов")

# Подготовить данные для оценки
eval_data = [
    {
        "inputs": {
            "query": query,
            "files": list(generated_files.keys()),
        },
        "outputs": {
            "response": all_code,
        }
    }
]

print("\n✓ Данные готовы к оценке")


ПОДГОТОВКА ДАННЫХ ДЛЯ СУДЕЙ

📂 Папка: ../scripts/demo_08
  ✓ index.html           |    886 символов
  ✓ main.js              |   3127 символов
  ✓ style.css            |   2281 символов
  ✓ styles.css           |   1042 символов

✓ Всего файлов: 4
✓ Общий размер кода: 7,894 символов

✓ Данные готовы к оценке


In [60]:
# Инициализировать модель
judge_model = ChatOpenAI(
    base_url=os.getenv("OPENAI_API_HOST"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.7,
)

print("✓ Модель инициализирована")

✓ Модель инициализирована


In [61]:
print("\n" + "="*100)
print("ЗАПУСК ОЦЕНКИ С ИСПОЛЬЗОВАНИЕМ LLM СУДЕЙ")
print("="*100)

with mlflow.start_run(run_name="llm_evaluation"):

    mlflow.log_param("evaluation_type", "llm_based")
    mlflow.log_param("model", "openai/gpt-oss-20b")
    mlflow.log_param("files_count", len(generated_files))
    mlflow.log_param("total_code_size", len(all_code))

    try:
        results = mlflow.genai.evaluate(
            data=[
                {
                    "inputs": {"query": query},
                    "outputs": {
                        "response": "\n".join(generated_files.values()),
                        "files": generated_files,
                    }
                }
            ],
            scorers=LLM_JUDGES
        )

        # ── Числовые метрики ──────────────────────────────────────────────────
        print("\n✅ ОЦЕНКА УСПЕШНО ЗАВЕРШЕНА!")
        print("\n📊 РЕЗУЛЬТАТЫ (числа):")
        if hasattr(results, 'metrics') and results.metrics:
            for metric_name, value in results.metrics.items():
                if isinstance(value, (int, float)) and 0 <= value <= 1:
                    print(f"  {metric_name:.<50} {value:.0%}")
                else:
                    print(f"  {metric_name:.<50} {value}")

        # ── Объяснения судьи (rationale) ──────────────────────────────────────
        print("\n📋 ОБЪЯСНЕНИЯ СУДЬИ:")
        if hasattr(results, 'tables') and 'eval_results' in results.tables:
            df = results.tables['eval_results']

            for col in df.columns:
                if col in ['inputs', 'outputs']:
                    continue

                value = df.iloc[0][col]

                if value is None or str(value) in ('nan', '', 'None'):
                    continue

                print(f"\n  {'='*60}")
                print(f"  📌 {col}")
                print(f"     {str(value)[:600]}")
        else:
            print(f"  Доступные таблицы: {list(results.tables.keys()) if hasattr(results, 'tables') else 'нет'}")

        print("\n✅ Результаты сохранены в MLflow!")
        print("   https://mlflow.aicorex.tech")

    except Exception as e:
        print(f"\n❌ ОШИБКА: {e}")
        import traceback
        traceback.print_exc()


ЗАПУСК ОЦЕНКИ С ИСПОЛЬЗОВАНИЕМ LLM СУДЕЙ


Evaluating: 100%|█| 1/1 [Elapsed: 00:11, Remaining: 00:00] [predict_fn: 0%, scor



✅ ОЦЕНКА УСПЕШНО ЗАВЕРШЕНА!

📊 РЕЗУЛЬТАТЫ (числа):
  agent_timing_metric/mean.......................... 50%
  test_results_metric/mean.......................... 100%
  llm_functionality_check/mean...................... 0%
  llm_architecture_assessment/mean.................. 40%
  webpage_quality_assessment/mean................... 34%
  expert_code_review/mean........................... 65%
  llm_overall_assessment/mean....................... 50%
  docker_build_assessment/mean...................... 84%

📋 ОБЪЯСНЕНИЯ СУДЬИ:

  📌 trace_id
     tr-d4529d04be3d469e084a7c733bc07e80

  📌 agent_timing_metric/value
     0.5

  📌 test_results_metric/value
     1.0

  📌 llm_functionality_check/value
     0.0

  📌 llm_architecture_assessment/value
     0.4

  📌 webpage_quality_assessment/value
     0.34

  📌 expert_code_review/value
     0.65

  📌 llm_overall_assessment/value
     0.5

  📌 docker_build_assessment/value
     0.843

  📌 trace
     {"info": {"trace_id": "tr-d4529d04be3d469e084a7c733